<a href="https://colab.research.google.com/github/RoshanHelmy/FlyRank-Assignment1-/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


One row represents the daily performance of one content page for one client, identified by `content_hash_id`, `client_hash_id`, and `report_date`.

For my Lane 2: Refresh / Content Opportunity Scoring project, I am using the March 2026 data as the development window. The dataset contains daily observations from this month, which allows me to study page performance using information available during that period.

In [6]:
from huggingface_hub import login, list_repo_files
import os

# Log in using the token stored in Colab Secrets
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

# Show the files available in the internship warehouse
files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Available files:")
for file in files:
    print(file)

Available files:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily

In [7]:
from huggingface_hub import hf_hub_download
import pandas as pd

# Download the March 2026 performance data
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

# Load the data
df = pd.read_parquet(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

First 5 rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Verify the unit of analysis and time window

print("Number of rows:", len(df))
print("Number of unique clients:", df["client_hash_id"].nunique())
print("Number of unique content pages:", df["content_hash_id"].nunique())
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

# Check whether client + content + date uniquely identifies a row
duplicate_keys = df.duplicated(
    subset=["client_hash_id", "content_hash_id", "report_date"]
).sum()

print("Duplicate client-content-date rows:", duplicate_keys)

Number of rows: 9841378
Number of unique clients: 55
Number of unique content pages: 331437
Date range: 2026-03-01 to 2026-03-31
Duplicate client-content-date rows: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


### Features
- `gsc_impressions` — search visibility available at the decision moment.
- `gsc_clicks` — search clicks available at the decision moment.
- `gsc_avg_position` — average search position available at the decision moment.
- `ga4_pageviews` — page views available at the decision moment when GA4 data is available.
- `scroll_events` — engagement information available at the decision moment when available.

### Label
- `trend_direction` — the target I want to predict or rank for: whether a content page is declining. This field is not present in the daily warehouse table, so it will need to be constructed from performance over time.

### Context
- `client_hash_id` — identifies the client.
- `content_hash_id` — identifies the content page.
- `report_date` — identifies when the observation was recorded.
- `gsc_data_available` — indicates whether GSC data is available.
- `ga4_data_available` — indicates whether GA4 data is available.

### Excluded
- `gsc_sum_position` — excluded because it is an aggregate measure that will be replaced by the more directly interpretable `gsc_avg_position`.
- `client_has_gsc` and `client_has_ga4` — excluded as model features because they describe client-level availability rather than page performance.
- Future performance or any field derived from the outcome — excluded to prevent target leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Verify the grain
The expected grain is one row per client, content page, and report date. I will verify that the combination of `client_hash_id`, `content_hash_id`, and `report_date` contains no duplicate rows.

### Query 2 — Verify row count and date span

The March 2026 development slice contains daily content performance observations. I will verify the number of rows and the date range.

### Query 3 — Verify data availability

The warehouse includes availability flags for GSC and GA4. I will check how many rows have GSC and GA4 data available. This is important because not every client has every data source available, so feature availability must be considered when building the model.

In [9]:
# Query 1: Verify the grain

duplicate_count = df.duplicated(
    subset=["client_hash_id", "content_hash_id", "report_date"]
).sum()

print("Duplicate client-content-date rows:", duplicate_count)

if duplicate_count == 0:
    print("Verified: each row represents one client-content-date observation.")
else:
    print("Warning: duplicate client-content-date observations exist.")

Duplicate client-content-date rows: 0
Verified: each row represents one client-content-date observation.


In [10]:
# Query 2: Verify row count and date span

print("Row count:", len(df))
print("Minimum report date:", df["report_date"].min())
print("Maximum report date:", df["report_date"].max())

Row count: 9841378
Minimum report date: 2026-03-01
Maximum report date: 2026-03-31


In [11]:
# Query 3: Verify availability
# Equivalent to SQL: WHERE gsc_data_available IS TRUE

gsc_available = df["gsc_data_available"].eq(True).sum()
ga4_available = df["ga4_data_available"].eq(True).sum()

print("Rows with GSC data available (IS TRUE):", gsc_available)
print("Rows with GA4 data available (IS TRUE):", ga4_available)

Rows with GSC data available (IS TRUE): 3611061
Rows with GA4 data available (IS TRUE): 413966


### Five initial features

For my Lane 2: Refresh / Content Opportunity Scoring project, I will start with five observable features:

1. `gsc_impressions` — available when GSC data is available, and represents search visibility at the decision moment.
2. `gsc_clicks` — available when GSC data is available, and represents the clicks received from search.
3. `gsc_avg_position` — available when GSC data is available, and represents the page's average search position.
4. `ga4_pageviews` — available when GA4 data is available, and represents page views from analytics data.
5. `scroll_events` — available when engagement tracking is available, and represents observed user engagement with the page.

These features are intended to describe the page's observable performance before deciding which content deserves review. Future performance used to evaluate the outcome will not be included as an input feature.

In [12]:
# Build a five-feature frame for Lane 2

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "scroll_events"
]

feature_frame = df[
    ["client_hash_id", "content_hash_id", "report_date"] + feature_cols
].copy()

print("Five-feature frame shape:", feature_frame.shape)

display(feature_frame.head(10))

Five-feature frame shape: (9841378, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,NaN,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,NaN,NaN
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,NaN,NaN
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,NaN,NaN
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,NaN,NaN
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,NaN,NaN
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,NaN,NaN
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,NaN,NaN
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,NaN,NaN
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,NaN,NaN


### The leakage trap

To demonstrate target leakage, I will deliberately create a feature using future information: the next month's GSC impressions.

This feature is not available at the decision moment because it comes from a future period. It could make a model appear stronger because it contains information about what happens after the decision.

I will use it only to demonstrate the leakage problem. It will not be kept in the final feature set.

In [13]:
# Deliberate leakage experiment:
# Use April 2026 impressions as a feature for March 2026 observations.

april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df_april = pd.read_parquet(april_path)

# Aggregate April impressions by client and content page
future_impressions = (
    df_april
    .groupby(["client_hash_id", "content_hash_id"])["gsc_impressions"]
    .sum()
    .reset_index()
    .rename(columns={"gsc_impressions": "future_gsc_impressions_leak"})
)

# Join future information onto March data
leakage_frame = df[
    ["client_hash_id", "content_hash_id", "report_date", "gsc_impressions"]
].copy()

leakage_frame = leakage_frame.merge(
    future_impressions,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Leakage feature created:", "future_gsc_impressions_leak" in leakage_frame.columns)
display(leakage_frame.head())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Leakage feature created: True


,client_hash_id,content_hash_id,report_date,gsc_impressions,future_gsc_impressions_leak
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,1151.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,73.0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,6787.0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,98.0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,405.0


### Leakage removed

The future GSC impressions feature is removed from the final feature set because it would not be known when the refresh-prioritization decision is made.

The final feature set contains only information that could be observed at or before the decision moment. The future data is reserved for evaluating outcomes rather than making the initial decision.

In [14]:
# Remove the deliberately leaked future feature

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "scroll_events"
]

final_feature_frame = df[
    ["client_hash_id", "content_hash_id", "report_date"] + honest_features
].copy()

print("Final honest feature columns:")
print(honest_features)

display(final_feature_frame.head())

Final honest feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'scroll_events']


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,NaN,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,NaN,NaN
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,NaN,NaN
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,NaN,NaN
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,NaN,NaN


### Leakage scoring experiment

To make the leakage effect measurable, I define a simple proxy label for demonstration: whether a page had any GSC impressions during the March 2026 observation window.

I then compare a deliberately leaked feature, `future_gsc_impressions_leak`, with an honest feature, `gsc_impressions`.

The leaked feature uses April 2026 information, which would not have been available when making a decision during March. Because it contains information from after the decision moment, it can create an artificially strong result.

This experiment is only a demonstration of leakage. The leaked feature is removed from the final feature set.

In [15]:
from sklearn.metrics import roc_auc_score

# Create a simple proxy label for the leakage demonstration:
# 1 = page had GSC impressions in March
# 0 = page had no GSC impressions in March
leakage_frame["has_march_impressions"] = (
    leakage_frame["gsc_impressions"].fillna(0) > 0
).astype(int)

# Remove rows where the label cannot be meaningfully evaluated
eval_df = leakage_frame.dropna(
    subset=["future_gsc_impressions_leak", "gsc_impressions"]
).copy()

y = eval_df["has_march_impressions"]

# Score using the honest March feature
honest_score = eval_df["gsc_impressions"].fillna(0)

# Score using the deliberately leaked April feature
leaked_score = eval_df["future_gsc_impressions_leak"].fillna(0)

honest_auc = roc_auc_score(y, honest_score)
leaked_auc = roc_auc_score(y, leaked_score)

print(f"Honest feature AUC: {honest_auc:.3f}")
print(f"Leaked future feature AUC: {leaked_auc:.3f}")

Honest feature AUC: 1.000
Leaked future feature AUC: 0.951


### Leakage experiment result

The leakage experiment demonstrates why future information must not be used as a model feature.

The `future_gsc_impressions_leak` feature comes from April 2026, after the March decision window. Even if it produces a stronger score than the honest March feature, that performance cannot be trusted because the information would not have been available at decision time.

The leaked feature is therefore removed. The final model will use only features that were observable at or before the decision moment, while future data will be reserved for evaluating the outcome.

This is a methodological demonstration of leakage rather than evidence that future impressions can genuinely predict past performance.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This data has several limitations that affect what the project can conclude.

First, the history is not necessarily balanced across all clients and content pages. Some pages or clients may have more observations than others.

Second, GSC and GA4 data are not available for every row. This means that some features may be missing or unavailable for certain pages, and availability itself needs to be considered when building the feature set.

Third, the daily performance data alone does not directly provide a ready-made `trend_direction` label. A future-performance proxy must be constructed from observations across time.

Finally, the data can show observed relationships between performance signals and future outcomes, but it cannot prove that refreshing a page will cause rankings, clicks, or traffic to improve. The output should therefore be treated as directional decision support rather than causal proof.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.